In [10]:
import os, warnings
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
warnings.filterwarnings(
    "ignore",
    category=FutureWarning,
    module="keras.src.export.tf2onnx_lib"
)
import tensorflow as tf
import pandas as pd
from tensorflow import keras
from tensorflow.keras.layers import StringLookup
from tensorflow.keras.saving import load_model
import cv2
import numpy as np
import matplotlib.pyplot as plt
from src.constants import *
from src.helpers import load_img, decode_true_labels, build_dataset, decode_model_output
from sklearn.model_selection import train_test_split
from tensorflow.keras.backend import ctc_batch_cost, int_shape, ctc_decode
import editdistance

In [3]:
df = pd.read_csv(TEST_TSV, sep='\t', header=None, names=['file', 'label'])
df = df.dropna(subset=['label']).reset_index(drop=True)
image_paths = [TEST_DIR + "/" + f for f in df["file"].values]
labels = df["label"].values

In [4]:
chars = []
with open("chars.txt") as file:
    line = file.readline()
    chars = list(line)
print(chars)

label_to_int = StringLookup(vocabulary=chars, num_oov_indices=0) # encode
label_to_str = StringLookup(vocabulary=label_to_int.get_vocabulary(), invert=True) # decode

['Т', 'К', 'с', 'Е', 'u', 'ъ', 'х', '+', ')', 'o', 'Н', '1', 'С', 'Г', 's', 'Я', '4', '"', 'н', 'з', '5', 'ю', 'ш', 'б', 'У', 'э', 'R', 'О', 'к', 't', 'Х', 'Ю', 'Ф', 'ч', 'Д', '[', 'В', '№', 'ф', 'r', 'З', 'b', "'", 'м', 'И', '9', 'Й', '?', 'П', '/', ';', 'ц', '2', '6', 'р', '.', 'c', '!', 'ы', 'в', '=', 'Б', 'p', 'ь', 'д', 'Э', 'у', '%', 'о', 'т', 'ж', 'Ч', ']', 'п', 'e', 'y', 'Р', 'ё', 'Л', '7', 'М', ':', ' ', 'й', 'а', 'Ж', '«', 'Щ', 'е', 'Ц', '(', '»', 'А', 'и', '0', 'я', 'г', '-', 'л', 'щ', ',', 'h', 'x', 'i', '3', '8', 'Ш']


I0000 00:00:1769181817.632645  136478 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 6053 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 2080 SUPER, pci bus id: 0000:01:00.0, compute capability: 7.5


In [5]:
class CTCLayer(tf.keras.layers.Layer):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.loss_func = ctc_batch_cost

    def call(self, y_true, y_pred):
        batch_size = tf.cast(tf.shape(y_true)[0], tf.int32)
        input_len = tf.cast(tf.shape(y_pred)[1], tf.int32)
        input_len *= tf.ones((batch_size, 1), tf.int32)

        label_len = tf.math.count_nonzero(y_true, axis=1, keepdims=True)
        
        loss = self.loss_func(y_true, y_pred, input_len, label_len)
        self.add_loss(loss)
        return y_pred
    
    def get_config(self):
        config = super().get_config()
        config.update({
            "loss_func": self.loss_func,
        })
        return config

In [6]:
model = load_model("modelFULL.keras", custom_objects={"CTCLayer": CTCLayer})

In [ ]:
test_ds = build_dataset(image_paths, labels, 32, label_to_int)
pred_y = model.predict(test_ds)

49/49 ━━━━━━━━━━━━━━━━━━━━ 4s 49ms/step


In [ ]:
truths = []
preds = []
preds_probs = []
for imgs, labels in test_ds:
    pred = model(imgs, training=False)
    preds_probs.append(pred)
    preds.extend(decode_model_output(pred, label_to_str))
    for seq in labels.numpy():
        seq = seq[seq != 0]
        truths.append(b"".join(label_to_str(seq).numpy()).decode("utf-8"))

# Character error rate (average levenshtein distance, the lower the better)
total_edits = 0
total_chars = 0
for p, t in zip(preds, truths):
    total_edits += editdistance.eval(p, t)
    total_chars += len(t)
cer = total_edits / max(1, total_chars)

# Exact word match (the higher the better)
correct = sum(p == t for p, t in zip(preds, truths))
ewm = correct / max(1, len(truths))

# Blank ratio (how many characters are being predicted as blanks)
# (generally the lower the better)
logits = tf.concat(preds_probs, axis=0)
pred_ids = tf.argmax(logits, axis=-1)
blank_idx = logits.shape[-1] - 1
blanks = tf.equal(pred_ids, blank_idx)
blank_cnt = tf.reduce_sum(tf.cast(blanks, tf.float32))
total_steps = tf.cast(tf.size(pred_ids), tf.float32)
blank_ratio = (blank_cnt / tf.maximum(1., total_steps)).numpy()

# Length ratio (how long the predicted label is vs. the actual length)
# (1.0 is the ideal value)
pred_lengths = [len(p) for p in preds]
truth_lengths = [len(t) for t in truths]
len_ratio = sum(pred_lengths) / max(1, sum(truth_lengths))

print(f"Cer: {cer}") # not good, but somewhat workable
print(f"Ewm: {ewm}")
print(f"Blank ratio: {blank_ratio}")
print(f"Length ratio: {len_ratio}")

Cer: 0.5657335581787521
Ewm: 0.03562176165803109
Blank ratio: 0.8150777220726013
Length ratio: 0.8956492411467116
